<style>
:root { --text:#1f2937; --muted:#5b6472; --line:#d7dde6; --soft:#f6f8fb; --accent:#274060; }
body { color: var(--text); line-height: 1.55; }
.jp-Notebook, .notebook-container { max-width: 1120px; margin: 0 auto; }
h1 { border-bottom: 2px solid var(--accent); padding-bottom: .25em; }
h2 { margin-top: 1.9em; border-bottom: 1px solid var(--line); padding-bottom: .2em; }
h3 { margin-top: 1.4em; }
table { border-collapse: collapse; margin-top: .5em; margin-bottom: 1em; }
th, td { border: 1px solid var(--line); padding: 6px 10px; vertical-align: top; }
th { background: var(--soft); }
code { font-size: 0.95em; }
blockquote { border-left: 4px solid var(--line); padding-left: 1em; color: var(--muted); }
</style>
# Z01. Wyrażenia regularne i ekstrakcja informacji

Dane znajdują się w pliku `Z01_logi.txt`. Każdy wiersz reprezentuje jeden komunikat z logu aplikacyjnego.

Celem jest zbudowanie zestawu funkcji, które przekształcą półustrukturyzowany tekst w dane o ustalonym schemacie. W rozwiązaniach należy rozdzielać:

1. dopasowanie struktury,
2. ekstrakcję wartości,
3. konwersję typu,
4. walidację semantyczną,
5. kontrolę jakości wyniku.

Dopasowanie jednego przykładu nie wystarcza do uznania wzorca za poprawny. Reguły należy sprawdzać na różnych rekordach i przypadkach brzegowych.


In [2]:
from pathlib import Path
import re
from collections import Counter
from statistics import median

logs = Path("Z01_logi.txt").read_text(encoding="utf-8").splitlines()
print("records:", len(logs))
print(logs[0])

records: 36
2026-09-25T10:14:03Z level=INFO service=auth ip=10.2.4.8 method=POST path=/login status=200 latency=84ms request=req-7F2A91


## 1. Rozpoznanie struktury

Wyświetl trzy pierwsze rekordy. Na ich podstawie wypisz pola występujące w logach i zapisz oczekiwany format każdego pola.

Następnie sprawdź cały plik i wskaż co najmniej dwa przypadki, które mogą wymagać dodatkowej walidacji lub normalizacji.

In [3]:
for line in logs[:3]:
    print(line)

# Przegląd całego pliku, bez zakładania poprawności wartości.
raw_records = [dict(token.split("=", 1) for token in line.split()[1:]) for line in logs]
for name in raw_records[0]:
    print(f"{name}: {sorted({r.get(name) for r in raw_records})}")

2026-09-25T10:14:03Z level=INFO service=auth ip=10.2.4.8 method=POST path=/login status=200 latency=84ms request=req-7F2A91
2026-09-25T10:14:04Z level=INFO service=catalog ip=10.2.4.19 method=GET path=/products/184 status=200 latency=41ms request=req-7F2A92
2026-09-25T10:14:05Z level=WARN service=catalog ip=10.2.4.19 method=GET path=/products/9999 status=404 latency=57ms request=req-7F2A93
level: ['ERROR', 'INFO', 'WARN']
service: ['auth', 'catalog', 'orders', 'payments']
ip: ['10.10.0.2', '10.10.0.3', '10.10.0.4', '10.2.4.19', '10.2.4.51', '10.2.4.60', '10.2.4.8', '10.2.8.10', '10.2.8.11', '10.2.8.12', '10.2.8.13', '10.2.8.14', '10.2.8.15', '10.2.8.7', '10.2.8.8', '10.2.8.9', '172.16.8.15', '172.16.8.4', '172.20.1.2', '172.20.1.3', '172.20.1.4', '172.20.1.5', '172.20.1.6', '172.20.1.7', '172.20.1.8', '172.20.1.9', '192.168.1.20', '192.168.1.41', '192.168.2.10', '192.168.2.9', '300.2.4.8', '999.1.1.1']
method: ['GET', 'PATCH', 'POST']
path: ['/categories/12', '/categories/14', '/login'

Pola i oczekiwane formaty:

| Pole | Format |
|---|---|
| timestamp | `YYYY-MM-DDTHH:MM:SSZ`, poprawna data i czas w UTC |
| level | `INFO`, `WARN` lub `ERROR` |
| service | małe litery `a-z` |
| ip | cztery grupy 1–3 cyfr, wartości 0–255 |
| method | metoda HTTP zapisana wielkimi literami, np. `GET`, `POST`, `PATCH` |
| path | ścieżka zaczynająca się `/`, bez białych znaków|
| status | trzy cyfry, kod HTTP w zakresie 100–599 |
| latency | nieujemna liczba całkowita lub dziesiętna i jednostka `ms` lub `s` |
| request | `req-` i dokładnie sześć znaków `0-9A-F` |

Przegląd całego pliku ujawnia adresy `300.2.4.8` i `999.1.1.1`, które wymagają walidacji zakresów. Opóźnienia wymagają przeliczenia sekund na milisekundy, a numeryczne segmenty ścieżek -
normalizacji do `{id}` z zachowaniem parametrów zapytania. Niepoprawne lub brakujące pola reprezentujemy jako `None`; adres o poprawnym formacie zachowujemy również wtedy, gdy `ip_valid` wynosi `False`.

## 2. Poziom logu i nazwa usługi

Napisz funkcje:

```python
extract_level(text)
extract_service(text)
```

`level` może przyjmować wyłącznie `INFO`, `WARN` albo `ERROR`. `service` składa się z małych liter.

Policz liczbę rekordów dla każdego poziomu i każdej usługi.

In [4]:
def extract_field(text, name, pattern):
    """Zwróć całe pole zgodne ze wzorcem albo None (bez dopasowań prefiksów)."""
    match = re.search(r"(?<!\S)" + re.escape(name) + r"=(" + pattern + r")(?=\s|$)", text)
    return match.group(1) if match else None


def extract_level(text):
    return extract_field(text, "level", r"INFO|WARN|ERROR")


def extract_service(text):
    return extract_field(text, "service", r"[a-z]+")


print("Poziomy:", Counter(extract_level(line) for line in logs))
print("Usługi:", Counter(extract_service(line) for line in logs))
assert extract_level("level=INFO_EXTRA") is None
assert extract_service("service=Auth") is None

Poziomy: Counter({'INFO': 22, 'WARN': 7, 'ERROR': 7})
Usługi: Counter({'catalog': 12, 'orders': 11, 'auth': 7, 'payments': 6})


## 3. Adres IPv4: format a poprawność wartości

Najpierw wyodrębnij napis mający strukturę czterech grup od jednej do trzech cyfr rozdzielonych kropkami.

Następnie napisz:

```python
valid_ipv4(value)
```

Funkcja ma zwracać `True` tylko wtedy, gdy wszystkie cztery liczby należą do przedziału od 0 do 255.

Sprawdź poniższe przypadki:

In [5]:
IP_PATTERN = r"[0-9]{1,3}(?:\.[0-9]{1,3}){3}"


def extract_ip(text):
    return extract_field(text, "ip", IP_PATTERN)


def valid_ipv4(value):
    if not isinstance(value, str) or re.fullmatch(IP_PATTERN, value) is None:
        return False
    return all(0 <= int(octet) <= 255 for octet in value.split("."))


assert valid_ipv4("10.2.4.8") is True
assert valid_ipv4("255.255.255.255") is True
assert valid_ipv4("300.2.4.8") is False
assert valid_ipv4("999.1.1.1") is False
assert valid_ipv4("10.2.4") is False
assert valid_ipv4("0.0.0.0") is True
assert valid_ipv4("1.2.3.4.5") is False
assert valid_ipv4("-1.2.3.4") is False
assert valid_ipv4(None) is False
print("Adresy o poprawnej strukturze, lecz błędnych oktetach:")
print([ip for line in logs if (ip := extract_ip(line)) is not None and not valid_ipv4(ip)])

Adresy o poprawnej strukturze, lecz błędnych oktetach:
['300.2.4.8', '999.1.1.1']


W 2–3 zdaniach wyjaśnij, dlaczego wygodnie jest oddzielić sprawdzenie struktury od sprawdzenia zakresów liczbowych.

Odpowiedź: Regex sprawdza strukturę napisu, a konwersja oktetów na liczby umożliwia proste sprawdzenie zakresu 0–255. Dzięki temu wzorzec pozostaje czytelny i można odróżnić brak lub błędny format pola od adresu o poprawnej strukturze, ale niepoprawnych wartościach. Zgodnie z poleceniem nie odrzucamy zer wiodących, jeśli oktety spełniają wymagania długości i zakresu.

## 4. Opóźnienie i jednostki

Pole `latency` występuje w milisekundach (`ms`) albo sekundach (`s`). Napisz funkcję:

```python
extract_latency_ms(text)
```

która zwraca opóźnienie zawsze w milisekundach jako `float`.

Przykłady:

- `latency=84ms` → `84.0`,
- `latency=0.42s` → `420.0`,
- `latency=1.8s` → `1800.0`.

Przyjmij, że liczba jest nieujemna.

In [6]:
def extract_latency_ms(text):
    value = extract_field(text, "latency", r"[0-9]+(?:\.[0-9]+)?(?:ms|s)")
    if value is None:
        return None
    if value.endswith("ms"):
        return float(value[:-2])
    return float(value[:-1]) * 1000.0


assert extract_latency_ms("latency=84ms") == 84.0
assert extract_latency_ms("latency=0.42s") == 420.0
assert extract_latency_ms("latency=1.8s") == 1800.0
assert extract_latency_ms("latency=0ms") == 0.0
assert extract_latency_ms("latency=-1ms") is None
assert extract_latency_ms("latency=12msXYZ") is None
print([extract_latency_ms(line) for line in logs])

[84.0, 41.0, 57.0, 420.0, 1800.0, 96.0, 65.0, 123.0, 110.0, 920.0, 38.0, 310.0, 73.0, 29.0, 1150.0, 77.0, 54.0, 45.0, 208.0, 2400.0, 115.0, 101.0, 62.0, 88.0, 950.0, 37.0, 83.0, 71.0, 104.0, 49.0, 1300.0, 280.0, 55.0, 31.0, 139.0, 880.0]


## 5. Kod HTTP

Wyodrębnij `status` jako liczbę całkowitą. Następnie przypisz kategorię:

- `2xx` — sukces,
- `4xx` — błąd klienta,
- `5xx` — błąd serwera.

Policz liczebności kategorii w całym pliku.

In [7]:
def extract_status(text):
    value = extract_field(text, "status", r"[0-9]{3}")
    if value is None:
        return None
    status = int(value)
    return status if 100 <= status <= 599 else None


def status_category(status):
    if status is None:
        return None
    return {2: "2xx", 4: "4xx", 5: "5xx"}.get(status // 100, "inne (1xx/3xx)")


print("Kategorie HTTP:", Counter(status_category(extract_status(line)) for line in logs))
assert extract_status("status=2000") is None
assert extract_status("status=999") is None

Kategorie HTTP: Counter({'2xx': 22, '4xx': 7, '5xx': 7})


## 6. Normalizacja ścieżek

Ścieżki zawierają identyfikatory zasobów, np.:

```text
/products/184
/orders/772/items
/payments/771
```

Napisz funkcję `normalize_path(path)`, która zamieni numeryczne identyfikatory na `{id}`:

```text
/products/{id}
/orders/{id}/items
/payments/{id}
```

Nie zmieniaj parametrów zapytania w `/search?q=...`.

Następnie policz najczęstsze znormalizowane ścieżki.

In [8]:
def extract_path(text):
    return extract_field(text, "path", r"/\S*")


def normalize_path(path):
    if path is None:
        return None
    pathname, separator, query = path.partition("?")
    pathname = re.sub(r"(?<=/)[0-9]+(?=/|$)", "{id}", pathname)
    return pathname + separator + query


assert normalize_path("/products/184") == "/products/{id}"
assert normalize_path("/orders/772/items") == "/orders/{id}/items"
assert normalize_path("/payments/771") == "/payments/{id}"
assert normalize_path("/search?q=123/456") == "/search?q=123/456"
assert normalize_path("/products/184?q=123") == "/products/{id}?q=123"
assert normalize_path("/products/a184") == "/products/a184"
print(Counter(normalize_path(extract_path(line)) for line in logs).most_common())

[('/products/{id}', 7), ('/payments/{id}', 6), ('/login', 5), ('/orders/{id}', 5), ('/orders', 4), ('/orders/{id}/items', 2), ('/logout', 2), ('/categories/{id}', 2), ('/search?q=headphones', 1), ('/search?q=keyboard', 1), ('/search?q=monitor', 1)]


## 7. Identyfikator żądania

Wyodrębnij pole `request`. Przyjmij, że poprawny identyfikator ma postać `req-` oraz sześć znaków z zakresu `0-9` i `A-F`.

Napisz dwa testy pozytywne i trzy testy negatywne dla wzorca.

In [9]:
REQUEST_PATTERN = r"req-[0-9A-F]{6}"


def extract_request(text):
    return extract_field(text, "request", REQUEST_PATTERN)


# Dwa testy pozytywne i trzy negatywne.
for value in ["req-7F2A91", "req-000ABC"]:
    assert re.fullmatch(REQUEST_PATTERN, value) is not None
    assert extract_request("request=" + value) == value
for value in ["req-7f2A91", "req-12345", "req-1234567"]:
    assert re.fullmatch(REQUEST_PATTERN, value) is None
    assert extract_request("request=" + value) is None
print([extract_request(line) for line in logs])

['req-7F2A91', 'req-7F2A92', 'req-7F2A93', 'req-7F2A94', 'req-7F2A95', 'req-7F2A96', 'req-7F2A97', 'req-7F2A98', 'req-7F2A99', 'req-7F2AA0', 'req-7F2AA1', 'req-7F2AA2', 'req-7F2AA3', 'req-7F2AA4', 'req-7F2AA5', 'req-7F2AA6', 'req-7F2AA7', 'req-7F2AA8', 'req-7F2AA9', 'req-7F2AB0', 'req-7F2AB1', 'req-7F2AB2', 'req-7F2AB3', 'req-7F2AB4', 'req-7F2AB5', 'req-7F2AB6', 'req-7F2AB7', 'req-7F2AB8', 'req-7F2AB9', 'req-7F2AC0', 'req-7F2AC1', 'req-7F2AC2', 'req-7F2AC3', 'req-7F2AC4', 'req-7F2AC5', 'req-7F2AC6']


## 8. Parser rekordu

Zbuduj funkcję `parse_log(text)`, która zwraca słownik:

```python
{
    "timestamp": ...,
    "level": ...,
    "service": ...,
    "ip": ...,
    "ip_valid": ...,
    "method": ...,
    "path": ...,
    "path_normalized": ...,
    "status": ...,
    "latency_ms": ...,
    "request": ...,
}
```

Wykorzystaj funkcje napisane we wcześniejszych zadaniach. Nie duplikuj logiki regexów w kilku miejscach.

In [10]:
from datetime import datetime


def extract_timestamp(text):
    match = re.match(r"[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}Z(?=\s|$)", text)
    if match is None:
        return None
    value = match.group()
    try:
        datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ")
    except ValueError:
        return None
    return value  # Zachowujemy zapis ISO 8601 w UTC po walidacji daty.


def extract_method(text):
    return extract_field(text, "method", r"GET|POST|PUT|PATCH|DELETE|HEAD|OPTIONS|CONNECT|TRACE")


def parse_log(text):
    ip = extract_ip(text)
    path = extract_path(text)
    return {
        "timestamp": extract_timestamp(text),
        "level": extract_level(text),
        "service": extract_service(text),
        "ip": ip,
        "ip_valid": valid_ipv4(ip),
        "method": extract_method(text),
        "path": path,
        "path_normalized": normalize_path(path),
        "status": extract_status(text),
        "latency_ms": extract_latency_ms(text),
        "request": extract_request(text),
    }


records = [parse_log(x) for x in logs]
assert parse_log(logs[0])["latency_ms"] == 84.0
assert parse_log(logs[0].replace("2026-09-25", "2026-02-30"))["timestamp"] is None
assert parse_log(logs[0].replace(" status=200", ""))["status"] is None
assert parse_log(logs[0].replace("ip=10.2.4.8", "ip=1.2.3"))["ip"] is None
records[:2]

[{'timestamp': '2026-09-25T10:14:03Z',
  'level': 'INFO',
  'service': 'auth',
  'ip': '10.2.4.8',
  'ip_valid': True,
  'method': 'POST',
  'path': '/login',
  'path_normalized': '/login',
  'status': 200,
  'latency_ms': 84.0,
  'request': 'req-7F2A91'},
 {'timestamp': '2026-09-25T10:14:04Z',
  'level': 'INFO',
  'service': 'catalog',
  'ip': '10.2.4.19',
  'ip_valid': True,
  'method': 'GET',
  'path': '/products/184',
  'path_normalized': '/products/{id}',
  'status': 200,
  'latency_ms': 41.0,
  'request': 'req-7F2A92'}]

## 9. Kontrola jakości parsera

Dla każdego pola policz liczbę braków. Osobno policz liczbę adresów IP, które mają poprawny format, ale niepoprawne wartości oktetów.

Wypisz wszystkie rekordy, dla których:

- `ip_valid` jest `False`, albo
- nie udało się wyodrębnić któregoś z wymaganych pól.

Krótko oceń, czy brak dopasowania oznacza błąd danych, błąd wzorca czy niepełną specyfikację.

In [11]:
required_fields = tuple(records[0])
missing = {field: sum(r[field] is None for r in records) for field in required_fields}
invalid_octets = sum(r["ip"] is not None and r["ip_valid"] is False for r in records)
print("Braki (None):", missing)
print("Poprawny format IP, błędne oktety:", invalid_octets)
for number, (line, record) in enumerate(zip(logs, records), start=1):
    if record["ip_valid"] is False or any(record[field] is None for field in required_fields):
        print(f"\nWiersz {number}: {line}")
        print(record)

# Kontrola pokrycia całego pliku; False nie jest brakiem wartości.
assert len(records) == len(logs) == 36
assert all(count == 0 for count in missing.values())
assert invalid_octets == 2

Braki (None): {'timestamp': 0, 'level': 0, 'service': 0, 'ip': 0, 'ip_valid': 0, 'method': 0, 'path': 0, 'path_normalized': 0, 'status': 0, 'latency_ms': 0, 'request': 0}
Poprawny format IP, błędne oktety: 2

Wiersz 10: 2026-09-25T10:14:12Z level=ERROR service=payments ip=300.2.4.8 method=POST path=/payments/772 status=500 latency=920ms request=req-7F2AA0
{'timestamp': '2026-09-25T10:14:12Z', 'level': 'ERROR', 'service': 'payments', 'ip': '300.2.4.8', 'ip_valid': False, 'method': 'POST', 'path': '/payments/772', 'path_normalized': '/payments/{id}', 'status': 500, 'latency_ms': 920.0, 'request': 'req-7F2AA0'}

Wiersz 28: 2026-09-25T10:14:30Z level=WARN service=auth ip=999.1.1.1 method=POST path=/login status=401 latency=71ms request=req-7F2AB8
{'timestamp': '2026-09-25T10:14:30Z', 'level': 'WARN', 'service': 'auth', 'ip': '999.1.1.1', 'ip_valid': False, 'method': 'POST', 'path': '/login', 'path_normalized': '/login', 'status': 401, 'latency_ms': 71.0, 'request': 'req-7F2AB8'}


Ocena jakości: nie ma braków dopasowania w tym pliku. Błędne oktety to błędy danych, mimo że regex poprawnie wyodrębnił adresy. W ogólnym przypadku brak dopasowania może oznaczać brak lub uszkodzenie pola, błąd wzorca albo poprawny wariant nieuwzględniony w specyfikacji; należy porównać surowy rekord z wymaganiami, zamiast automatycznie uznać dane za błędne.

## 10. Podsumowanie danych

Na podstawie sparsowanych rekordów oblicz:

1. liczbę rekordów według `service` i `level`,
2. medianę `latency_ms` według usługi,
3. udział statusów `5xx` według usługi,
4. trzy najczęstsze znormalizowane ścieżki.

Na końcu wskaż dwie informacje, których nie powinno się próbować wydobywać z tych logów samymi wyrażeniami regularnymi, gdyby wymagały interpretacji znaczenia komunikatu.

In [12]:
print("Liczebności według usługi i poziomu:")
service_level_counts = Counter((r["service"], r["level"]) for r in records)
for service in sorted({r["service"] for r in records if r["service"] is not None}):
    group = [r for r in records if r["service"] == service]
    latencies = [r["latency_ms"] for r in group if r["latency_ms"] is not None]
    statuses = [r["status"] for r in group if r["status"] is not None]
    counts = {level: service_level_counts[service, level] for level in ("INFO", "WARN", "ERROR")}
    latency_median = median(latencies) if latencies else None
    error_share = sum(status_category(s) == "5xx" for s in statuses) / len(statuses) if statuses else None
    print(f"{service}: {counts}, mediana={latency_median} ms, udział 5xx={error_share:.2%}")

# Udział 5xx liczony wśród poprawnie odczytanych statusów danej usługi.
# Rekordy z błędnym IP zachowujemy: ich status i opóźnienie są dostępne.
top_paths = Counter(r["path_normalized"] for r in records if r["path_normalized"] is not None).most_common(3)
print("Trzy najczęstsze ścieżki:", top_paths)

Liczebności według usługi i poziomu:
auth: {'INFO': 4, 'WARN': 3, 'ERROR': 0}, mediana=65.0 ms, udział 5xx=0.00%
catalog: {'INFO': 8, 'WARN': 3, 'ERROR': 1}, mediana=56.0 ms, udział 5xx=8.33%
orders: {'INFO': 8, 'WARN': 1, 'ERROR': 2}, mediana=208.0 ms, udział 5xx=18.18%
payments: {'INFO': 2, 'WARN': 0, 'ERROR': 4}, mediana=1110.0 ms, udział 5xx=66.67%
Trzy najczęstsze ścieżki: [('/products/{id}', 7), ('/payments/{id}', 6), ('/login', 5)]


Podsumowanie:

- Przetworzono 36 rekordów bez braków pól. Dwa adresy IP (`300.2.4.8` i `999.1.1.1`) mają poprawną strukturę, lecz niepoprawne wartości oktetów.
- Usługa `payments` ma największy udział odpowiedzi 5xx: 4/6, czyli 66,67%, oraz najwyższą medianę opóźnienia: 1110 ms. Są to obserwacje dla tego małego zbioru, a nie dowód przyczyny problemów.
- Sam regex nie pozwala ustalić **rzeczywistej przyczyny awarii** (np. problemu bazy danych lub zależnej usługi) ani **intencji użytkownika** (np. czy nieudane logowanie jest pomyłką czy próbą ataku). Wymaga to dodatkowego kontekstu i interpretacji; te logi akurat nie zawierają opisowych komunikatów umożliwiających takie rozstrzygnięcia.